## **Ridge Regularization From Scratch (Gradient Descent Method)**

### **Topic Roadmap**

 **1. Imports & Setup**

 **2. Dataset Preparation**

 **3. Scikit-Learn Baselines**

 **4. Custom Ridge Implementation (Gradient Descent)**

 **5. Model Evaluation**

### **1. Imports & Setup**

Import the required libraries for data handling, modeling, and evaluation[cite: 9].

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor, Ridge
from sklearn.metrics import r2_score

### **2. Dataset Preparation**

Load the diabetes dataset and split it into training and testing sets[cite: 9].

In [2]:
# Load dataset
X, y = load_diabetes(return_X_y=True)

# Create training and testing splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

### **3. Scikit-Learn Baselines**

Establish performance baselines using Scikit-Learn's built-in models. 

We use `SGDRegressor` configured with an $L2$ penalty, which mathematically equates to applying Gradient Descent to solve Ridge Regression[cite: 9].

In [3]:
# Initialize and train SGDRegressor with an L2 penalty (Ridge equivalent)
sgd_baseline = SGDRegressor(penalty='l2', max_iter=500, eta0=0.1, learning_rate='constant', alpha=0.001)
sgd_baseline.fit(X_train, y_train)

# Evaluate baseline performance
y_pred_sgd = sgd_baseline.predict(X_test)
print(f"Sklearn SGDRegressor (L2) R2 Score: {r2_score(y_test, y_pred_sgd):.4f}")

Sklearn SGDRegressor (L2) R2 Score: 0.4552


### **4. Custom Ridge Implementation (Gradient Descent)**

Build a custom Ridge regressor that updates weights iteratively using Gradient Descent[cite: 9].

The gradient of the Ridge loss function with respect to the weights is calculated and subtracted at each epoch, gradually minimizing both the error and the $L2$ magnitude of the coefficients[cite: 9].

In [4]:
class CustomRidgeGD:
    def __init__(self, epochs=500, learning_rate=0.005, alpha=0.001):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
        # Initialize coefficients to 1s and intercept to 0
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        
        # Combine intercept and coefficients into a single theta vector
        theta = np.insert(self.coef_, 0, self.intercept_)
        
        # Prepend a column of 1s to X_train to account for the intercept term
        X_train_bias = np.insert(X_train, 0, 1, axis=1)
        
        # Iteratively update weights using Gradient Descent
        for i in range(self.epochs):
            # Calculate the gradient: (X^T * X * theta) - (X^T * y) + (alpha * theta)
            theta_der = np.dot(X_train_bias.T, X_train_bias).dot(theta) - np.dot(X_train_bias.T, y_train) + (self.alpha * theta)
            
            # Apply parameter updates
            theta = theta - (self.learning_rate * theta_der)
        
        # Extract the final intercept and coefficients
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
    
    def predict(self, X_test):
        # Predict using the dot product of features and coefficients, plus the intercept
        return np.dot(X_test, self.coef_) + self.intercept_

### **5. Model Evaluation**

Train the custom Ridge model and evaluate its performance against the baseline[cite: 9].

In [5]:
# Initialize and train the custom model
custom_ridge_gd = CustomRidgeGD(epochs=500, alpha=0.001, learning_rate=0.005)
custom_ridge_gd.fit(X_train, y_train)

# Evaluate custom model performance
y_pred_custom = custom_ridge_gd.predict(X_test)

print(f"Custom Ridge GD R2 Score: {r2_score(y_test, y_pred_custom):.4f}")

Custom Ridge GD R2 Score: 0.4738


### **Key Revision Notes**

- **Gradient Formulation:** The Ridge regression gradient includes the regularization term `alpha * theta`. This continuously pulls the weights towards zero during optimization[cite: 9].
- **Intercept Handling in GD:** In rigorous implementations, the intercept should not be regularized. The simplified scratch version above includes the intercept in `theta` and regularizes it alongside the features[cite: 9]. While acceptable for a conceptual demonstration, production models isolate the intercept penalty.
- **Learning Rate Sensitivity:** When adapting Ridge to Gradient Descent, scaling down the `learning_rate` relative to the unregularized model is often necessary, as the added penalty increases the gradient magnitude, risking divergence.